Práctica guiada: Control de transacciones con TCL en SQLite

Crear la base de datos y la tabla

In [1]:
# ============================================
# PASO 1: CREACIÓN DE LA BASE DE DATOS
# ============================================

import sqlite3
import os
import pandas as pd

# Nombre de la base de datos
DB_NAME = "pipeline_tcl.db"

# Ruta para Google Colab
DB_PATH = os.path.join("/content", DB_NAME)

# Crear conexión con SQLite
conn = sqlite3.connect(DB_PATH)

# Crear cursor para ejecutar comandos SQL
cursor = conn.cursor()

print("Conexión establecida correctamente.")
print(f"Base de datos: {DB_PATH}")

Conexión establecida correctamente.
Base de datos: /content/pipeline_tcl.db


Crear la tabla

In [2]:
# ============================================
# CREACIÓN DE LA TABLA PIPELINE
# ============================================

# Creamos una tabla para simular un pipeline de datos.
# Cada registro representa un dato procesado.

cursor.execute("""
CREATE TABLE IF NOT EXISTS pipeline (
    id INTEGER PRIMARY KEY,
    nombre TEXT NOT NULL,
    estado TEXT NOT NULL,
    valor REAL,
    fecha TEXT
)
""")

# Confirmamos la creación de la tabla
conn.commit()

print("Tabla 'pipeline' creada correctamente.")

Tabla 'pipeline' creada correctamente.


Insertar datos iniciales

In [3]:
# ============================================
# PASO 2: INSERTAR DATOS INICIALES
# ============================================

datos_iniciales = [
    (1, "Cliente A", "procesado", 1500.00, "2026-07-24"),
    (2, "Cliente B", "procesado", 2300.00, "2026-07-24"),
    (3, "Cliente C", "pendiente", 1800.00, "2026-07-24")
]

cursor.executemany("""
INSERT INTO pipeline (id, nombre, estado, valor, fecha)
VALUES (?, ?, ?, ?, ?)
""", datos_iniciales)

# Confirmamos los datos iniciales
conn.commit()

print("Datos iniciales insertados correctamente.")

Datos iniciales insertados correctamente.


Verificacion de datos

In [4]:
# ============================================
# VERIFICAR DATOS
# ============================================

df = pd.read_sql_query(
    "SELECT * FROM pipeline",
    conn
)

display(df)

,id,nombre,estado,valor,fecha
0,1,Cliente A,procesado,1500.0,2026-07-24
1,2,Cliente B,procesado,2300.0,2026-07-24
2,3,Cliente C,pendiente,1800.0,2026-07-24


Iniciar una transacción

In [5]:
# ============================================
# PASO 3: INICIAR TRANSACCIÓN
# ============================================

# SQLite inicia automáticamente una transacción
# cuando realizamos una operación que modifica datos.
#
# En este caso vamos a insertar un nuevo registro.

cursor.execute("""
INSERT INTO pipeline (id, nombre, estado, valor, fecha)
VALUES (?, ?, ?, ?, ?)
""", (
    4,
    "Cliente D",
    "procesado",
    3200.00,
    "2026-07-24"
))

print("Registro del Cliente D insertado.")

Registro del Cliente D insertado.


Actualizar los registros

In [6]:
# ============================================
# ACTUALIZAR DATOS
# ============================================

# Cambiamos el estado del Cliente C
# de "pendiente" a "procesado".

cursor.execute("""
UPDATE pipeline
SET estado = ?
WHERE id = ?
""", (
    "procesado",
    3
))

print("Estado del Cliente C actualizado.")

Estado del Cliente C actualizado.


Crear un SAVEPOINT

In [7]:
# ============================================
# PASO 4: CREAR SAVEPOINT
# ============================================

# Creamos un punto de recuperación llamado
# "punto_intermedio".

cursor.execute("SAVEPOINT punto_intermedio")

print("SAVEPOINT creado correctamente.")

SAVEPOINT creado correctamente.


Realizar nuevos cambios después del SAVEPOINT

In [8]:
# ============================================
# INSERTAR REGISTRO POSTERIOR AL SAVEPOINT
# ============================================

cursor.execute("""
INSERT INTO pipeline (id, nombre, estado, valor, fecha)
VALUES (?, ?, ?, ?, ?)
""", (
    5,
    "Cliente E",
    "procesado",
    4100.00,
    "2026-07-24"
))

print("Cliente E insertado.")

Cliente E insertado.


Actualizamos los registros

In [9]:
# ============================================
# SEGUNDA ACTUALIZACIÓN
# ============================================

cursor.execute("""
UPDATE pipeline
SET valor = ?
WHERE id = ?
""", (
    9999.99,
    2
))

print("Valor del Cliente B actualizado.")

Valor del Cliente B actualizado.


In [10]:
# ============================================
# CONSULTAR ESTADO TEMPORAL
# ============================================

df = pd.read_sql_query(
    "SELECT * FROM pipeline",
    conn
)

display(df)

,id,nombre,estado,valor,fecha
0,1,Cliente A,procesado,1500.00,2026-07-24
1,2,Cliente B,procesado,9999.99,2026-07-24
2,3,Cliente C,procesado,1800.00,2026-07-24
3,4,Cliente D,procesado,3200.00,2026-07-24
4,5,Cliente E,procesado,4100.00,2026-07-24


Simular un error

In [11]:
# ============================================
# PASO 5: SIMULAR UN ERROR
# ============================================

try:

    # Intentamos insertar un ID duplicado.
    # El ID 2 ya existe en la tabla.

    cursor.execute("""
    INSERT INTO pipeline (id, nombre, estado, valor, fecha)
    VALUES (?, ?, ?, ?, ?)
    """, (
        2,
        "Cliente Error",
        "procesado",
        5000.00,
        "2026-07-24"
    ))

except sqlite3.IntegrityError as error:

    print("Se produjo un error controlado.")
    print(f"Detalle del error: {error}")

Se produjo un error controlado.
Detalle del error: UNIQUE constraint failed: pipeline.id


Aplicar ROLLBACK parcial

In [12]:
# ============================================
# PASO 6: ROLLBACK PARCIAL
# ============================================

# Volvemos al punto intermedio.
# Esto deshace las operaciones realizadas DESPUÉS
# del SAVEPOINT.

cursor.execute("""
ROLLBACK TO SAVEPOINT punto_intermedio
""")

print("ROLLBACK parcial ejecutado.")

ROLLBACK parcial ejecutado.


eliminamos el SAVEPOINT

In [13]:
# ============================================
# LIBERAR SAVEPOINT
# ============================================

cursor.execute("""
RELEASE SAVEPOINT punto_intermedio
""")

print("SAVEPOINT liberado.")

SAVEPOINT liberado.


Confirmar la transacción con COMMIT

In [14]:
# ============================================
# PASO 7: CONFIRMAR TRANSACCIÓN
# ============================================

# COMMIT confirma definitivamente todos los cambios
# que no fueron revertidos.

conn.commit()

print("Transacción confirmada correctamente.")

Transacción confirmada correctamente.


Verificar el resultado final

In [15]:
# ============================================
# PASO 8: VERIFICAR RESULTADO FINAL
# ============================================

df_final = pd.read_sql_query(
    "SELECT * FROM pipeline ORDER BY id",
    conn
)

display(df_final)

,id,nombre,estado,valor,fecha
0,1,Cliente A,procesado,1500.0,2026-07-24
1,2,Cliente B,procesado,2300.0,2026-07-24
2,3,Cliente C,procesado,1800.0,2026-07-24
3,4,Cliente D,procesado,3200.0,2026-07-24


Reflexión



Esta práctica demuestra cómo el control transaccional permite mantener la integridad y consistencia de los datos durante operaciones complejas.

El comando COMMIT confirma definitivamente los cambios realizados durante una transacción. Una vez ejecutado, las modificaciones quedan almacenadas en la base de datos.

Por otro lado, ROLLBACK permite cancelar los cambios de una transacción que todavía no fueron confirmados. En este ejercicio utilizamos ROLLBACK TO SAVEPOINT, lo que permitió realizar un rollback parcial, eliminando únicamente las modificaciones realizadas después del punto de guardado.

El SAVEPOINT resulta especialmente útil en procesos de datos donde existen varias etapas. Si ocurre un error en una etapa intermedia, no es necesario cancelar todo el proceso: podemos regresar al último punto seguro y continuar trabajando desde allí.

En un escenario real de Data Science o Data Engineering, este mecanismo podría utilizarse en un pipeline donde se cargan, transforman y actualizan datos. Si una transformación falla, el sistema puede revertir únicamente la parte problemática, evitando que información parcialmente procesada quede almacenada de forma incorrecta.


```
Resumen de los comandos TCL utilizados
Comando	Función	Impacto
SAVEPOINT	Crea un punto de recuperación	Permite controlar una parte de la transacción
ROLLBACK TO SAVEPOINT	Revierte cambios posteriores al punto	Realiza un rollback parcial
RELEASE SAVEPOINT	Elimina el punto de recuperación	Libera el savepoint
COMMIT	Confirma los cambios	Hace permanentes las operaciones
ROLLBACK	Cancela la transacción	Revierte cambios no confirmados
```


